In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
# Load datasets
df1 = pd.read_csv('Data/mitbih_train_processed.csv')
df2 = pd.read_csv('Data/mitbih_test_processed.csv')
X_train, y_train = df1.iloc[:, :-1], df1.iloc[:, -1]
X_test, y_test = df2.iloc[:, :-1], df2.iloc[:, -1]

In [3]:
Xtemp0 = X_test.iloc[0:10]
Xtemp1 = X_test.iloc[18320:18330]
Xtemp2 = X_test.iloc[19053:19063]
Xtemp3 = X_test.iloc[20156:20166]
Xtemp4 = X_test.iloc[20393:20403]

In [4]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 2, figsize=(15, 20))

for i in range(10):
    ax = axes[i // 2, i % 2]
    ax.plot(X_train.iloc[i, :])
    ax.set_title(f'Sample {i+1}')

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [5]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)



DecisionTreeClassifier(random_state=42)

In [6]:
print("Number of nodes before pruning: ", model.tree_.node_count)
print("Original accuracy: ", accuracy_score(y_test, model.predict(X_test)))

Number of nodes before pruning:  4711
Original accuracy:  0.9539993604677721


In [7]:
path = model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas  

optimal_alpha = ccp_alphas[len(ccp_alphas) // 2]  
optimal_alpha

2.2685010507312018e-05

In [8]:
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

pruned_tree = DecisionTreeClassifier(random_state=42, ccp_alpha=optimal_alpha)
pruned_tree.fit(X_train, y_train)

print("Number of nodes after pruning:", pruned_tree.tree_.node_count)
print("Accuracy after pruning:", pruned_tree.score(X_test, y_test))

# Additional classification metrics
y_pred = pruned_tree.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print("Classification Report:\n", report_df)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Save the pruned tree classifier
joblib.dump(pruned_tree, 'pruned_tree.pkl')

# Append 10 samples for each class (0, 1, 2, 3, 4) into their respective segments list


Number of nodes after pruning: 1955
Accuracy after pruning: 0.9612626193412818
Classification Report:
               precision    recall  f1-score       support
0              0.972938  0.986256  0.979552  18117.000000
1              0.756757  0.604317  0.672000    556.000000
2              0.891756  0.859116  0.875132   1448.000000
3              0.736842  0.604938  0.664407    162.000000
4              0.963320  0.930970  0.946869   1608.000000
accuracy       0.961263  0.961263  0.961263      0.961263
macro avg      0.864323  0.797119  0.827592  21891.000000
weighted avg   0.959624  0.961263  0.960100  21891.000000
Confusion Matrix:
 [[17868    92   101    15    41]
 [  209   336     8     2     1]
 [  164     7  1244    18    15]
 [   41     0    23    98     0]
 [   83     9    19     0  1497]]


['pruned_tree.pkl']

In [14]:
segments0 = []
segments1 = []
segments2 = []
segments3 = []
segments4 = []

def remove_trailing_zeros_and_last_40(sample):
    sample = sample[:(sample != 0).cumsum().argmax() + 1]
    return sample[:-40] if len(sample) > 40 else sample

for i in range(len(y_test)):
    if y_test.iloc[i] == y_pred[i]:
        sample = X_test.iloc[i]
        sample = remove_trailing_zeros_and_last_40(sample)
        if y_test.iloc[i] == 0 and len(segments0) < 10:
            segments0.append(sample)
        elif y_test.iloc[i] == 1 and len(segments1) < 10:
            segments1.append(sample)
        elif y_test.iloc[i] == 2 and len(segments2) < 10:
            segments2.append(sample)
        elif y_test.iloc[i] == 3 and len(segments3) < 10:
            segments3.append(sample)
        elif y_test.iloc[i] == 4 and len(segments4) < 10:
            segments4.append(sample)

print("Segments for class 0:", segments0)
print("Segments for class 1:", segments1)
print("Segments for class 2:", segments2)
print("Segments for class 3:", segments3)
print("Segments for class 4:", segments4)

Segments for class 0: [val1     3721
val2     3211
val3     2176
val4     1485
val5     1500
val6     1410
val7     1365
val8     1260
val9     1215
val10    1230
val11    1245
val12    1380
val13    1545
val14    1605
val15    1800
val16    1830
val17    1875
val18    1965
val19    2101
val20    2191
val21    2401
val22    2401
val23    2476
val24    2596
val25    2521
val26    2386
val27    2311
val28    2131
val29    2010
val30    2010
val31    1920
val32    1860
val33    1845
val34    1905
val35    1845
val36    1770
val37    1740
val38    1680
val39    1605
val40    1515
val41    1530
val42    1500
val43    1305
val44    1275
val45    1335
val46    1275
val47    1290
val48    1320
val49    1410
val50    1410
val51    1365
val52    1245
val53    1260
val54    1155
val55    1110
val56    1170
val57    1080
val58    1125
val59    1155
val60    1095
Name: 0, dtype: int64, val1     2990
val2      870
val3        0
val4      489
val5      417
         ... 
val65     797
val66     870
va

In [18]:
hybrids = {}
for i in range(5):
    hybrids[i] = []
for x in segments0 :
    hybrids[0].extend(x)
for x in segments1 :
    hybrids[1].extend(x)
for x in segments2 :
    hybrids[2].extend(x)
for x in segments3 :
    hybrids[3].extend(x)
for x in segments4 :
    hybrids[4].extend(x)


In [ ]:
tree = pruned_tree.tree_

def traverse_tree(tree, node=0, depth=1):
    indent = "    " * depth  # Indentation for readability
    if tree.feature[node] != -2:  # Not a leaf node
        feature = "features({})".format(tree.feature[node] + 1)  # Convert to 1-based index
        threshold = int(round(tree.threshold[node]))  # Round threshold to integer
        print("{}if {} <= {}".format(indent, feature, threshold))
        traverse_tree(tree, tree.children_left[node], depth + 1)
        print("{}else".format(indent))
        traverse_tree(tree, tree.children_right[node], depth + 1)
        print("{}end".format(indent))  # Closing the if block
    else:  # Leaf node
        print("{}output = {};".format(indent, tree.value[node].argmax()))

print("function output = decision_tree(features)")
traverse_tree(tree)
print("end")


function output = decision_tree(features)
    if features(5) <= 1610
        if features(37) <= 2256
            if features(1) <= 2385
                if features(2) <= 410
                    if features(99) <= 2312
                        if features(12) <= 1350
                            output = 2;
                        else
                            if features(4) <= 1064
                                output = 2;
                            else
                                if features(28) <= 3202
                                    output = 2;
                                else
                                    output = 0;
                                end
                            end
                        end
                    else
                        if features(113) <= 1454
                            if features(8) <= 824
                                output = 0;
                            else
                                output = 2;
       

In [ ]:
tree = pruned_tree.tree_

def traverse_tree(tree, node=0, depth=1):
    indent = "    " * depth  # Indentation for readability
    if tree.feature[node] != -2:  # Not a leaf node
        feature = f"features[{tree.feature[node]}]"  # Use array index notation
        threshold = int(round(tree.threshold[node]))  # Round threshold to integer
        print(f"{indent}if {feature} <= {threshold}:")
        traverse_tree(tree, tree.children_left[node], depth + 1)
        print(f"{indent}else:")
        traverse_tree(tree, tree.children_right[node], depth + 1)
    else:  # Leaf node
        print(f"{indent}return {tree.value[node].argmax()}")  

print("def decision_tree(features):")
traverse_tree(tree)


def decision_tree(features):
    if features[4] <= 1610:
        if features[36] <= 2256:
            if features[0] <= 2385:
                if features[1] <= 410:
                    if features[98] <= 2312:
                        if features[11] <= 1350:
                            return 2
                        else:
                            if features[3] <= 1064:
                                return 2
                            else:
                                if features[27] <= 3202:
                                    return 2
                                else:
                                    return 0
                    else:
                        if features[112] <= 1454:
                            if features[7] <= 824:
                                return 0
                            else:
                                return 2
                        else:
                            return 0
                else:
                    if feature